Chart Analysis
Use Python to retrieve stock market data and analyze price changes.
1. Fetch both daily and hourly stock market data using a suitable API.
Visualize the data in several types of plots.
2. Plot long-term stock data over multiple years.
Create an interactive graph that allows the user to view different time ranges.
3. Start with Apple (AAPL) and plot its stock chart using daily data over several years.
4. Add moving averages to the stock charts, such as:
○ SMA20 and SMA50, or
○ SMA50 and SMA150
These should be overlaid on the main stock price chart.
5. Create a table for five stocks. Use default examples such as FAANG stocks plus
NVIDIA, but allow the user to change them.
For each stock, calculate the historical probability that the next day is positive if the
current day changes by:
+2%, +3%, +4%, +5%, and +6%.
Example: if Apple rises 5% today, what is the historical probability that the next day
closes positive?
Use the last 2 years of historical data.
6. Repeat the same analysis for negative daily moves, such as:
-2%, -3%, -4%, and -5%.
7. Repeat both of the above analyses again using 5 years of historical data.
8. Create a comparison table for the five selected stocks.
Include a simple correlation / collinearity analysis between them.
9. Plot a comparison chart for two to five stocks using daily data for one year.
Example: compare Apple and Amazon.
Normalize or rescale prices if needed so they can be compared clearly.
10. Create another table for the same five stocks showing the probability that the next day
is positive after:
● 2 consecutive positive days
● 3 consecutive positive days
● 4 consecutive positive days
● 5 consecutive positive days
● 6 consecutive positive days
11. Repeat the same analysis for consecutive negative days.
12. Screen all Nasdaq and NYSE stocks and identify the top 10 trending stocks, ranked
by daily percentage change.
13. Repeat the screening, but only include stocks with a market capitalization of $5 billion
or more.
This may require obtaining market cap data from an API or web scraping source.

I will be using the Massive API, formerly polygon.io. Massive obtains its data from an aggregate of all major US stock exchanges. The plan I am selecting provides 5 years of historical data.

In [1]:
import requests
import pandas as pd
from pathlib import Path
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

# Bypass scientific notation
#pd.options.display.float_format = '{:,.0f}'.format


def load_api_key(filepath="api_keys/massive.txt"):
    """Load Massive API key from a local text file."""
    return Path(filepath).read_text(encoding="utf-8").strip()


def get_all_pages(url, params=None):
    """Fetch all paginated results from Massive."""
    all_results = []

    while url:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        if data.get("status") == "NOT_AUTHORIZED":
            raise PermissionError(data.get("message", "Not authorized for this request."))

        results = data.get("results", [])
        all_results.extend(results)

        # After the first request, next_url already contains the needed query info
        url = data.get("next_url")
        params = None

    return all_results


def get_massive_daily_bars(
    ticker="AAPL",
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc"
):
    """
    Fetch daily aggregate bars for a ticker from Massive.

    Parameters
    ----------
    ticker : str
        Stock ticker symbol, e.g. 'AAPL'
    api_key_path : str
        Path to text file containing API key
    years : int
        Number of years of history to request
    safety_days : int
        Number of days to move forward from exact cutoff to avoid entitlement edge issues
    adjusted : bool
        Whether to return adjusted prices
    sort : str
        'asc' or 'desc'

    Returns
    -------
    pd.DataFrame
    """
    api_key = load_api_key(api_key_path)

    today = date.today()
    start_date = today - relativedelta(years=years) + timedelta(days=safety_days)

    start_str = start_date.isoformat()
    end_str = today.isoformat()

    url = f"https://api.massive.com/v2/aggs/ticker/{ticker}/range/1/day/{start_str}/{end_str}"

    params = {
        "adjusted": str(adjusted).lower(),
        "sort": sort,
        "limit": 50000,
        "apiKey": api_key,
    }

    results = get_all_pages(url, params)

    if not results:
        raise ValueError(f"No data returned for {ticker}.")

    df = pd.DataFrame(results).rename(columns={
        "t": "timestamp",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
        "vw": "vwap",
        "n": "transactions",
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df["ticker"] = ticker

    preferred_order = [
        "timestamp", "ticker", "open", "high", "low", "close",
        "volume", "vwap", "transactions"
    ]
    df = df[[col for col in preferred_order if col in df.columns]]

    return df

def fetch_and_save_ticker(
    ticker,
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc",
    save_folder="raw_daily_data"
):
    df = get_massive_daily_bars(
        ticker=ticker,
        api_key_path=api_key_path,
        years=years,
        safety_days=safety_days,
        adjusted=adjusted,
        sort=sort
    )

    filename = f"{save_folder}/daily_{ticker.lower()}.csv"
    df.to_csv(filename, index=False)

    print(f"{ticker} saved to {filename}")
    return df


tickers = [
    # Big Tech / Growth
    # META/FB must be handled separately due to name change
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "TSLA", "NFLX",
    
    # Finance
    "JPM", "BAC", "GS", "MS",
    
    # Consumer / Retail
    "WMT", "COST", "HD", "NKE", "SBUX",
    
    # Healthcare / Pharma
    "JNJ", "PFE", "MRK", "UNH",
    
    # Energy
    "XOM", "CVX",
    
    # Industrials / Transportation
    "BA", "CAT", "GE", "UPS",
    
    # ETFs (nice for comparison)
    "SPY", "QQQ", "DIA"
]



# Maybe better structure, don't worry about it for now until I can do more testing
# tickers_by_sector = {
#     "Tech": ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "NFLX],
#     "Finance": ["JPM", "BAC", "GS", "MS"],
#     "Consumer": ["WMT", "COST", "HD", "NKE", "SBUX"],
#     "Healthcare": ["JNJ", "PFE", "MRK", "UNH"],
#     "Energy": ["XOM", "CVX"],
#     "Industrial": ["BA", "CAT", "GE", "UPS"],
#     "ETFs": ["SPY", "QQQ", "DIA"]
# }






data = {}

for t in tickers:
    data[t] = fetch_and_save_ticker(t)


# Special FB/META handling
# FB was Meta's old ticker through 2022-06-08
# META became the ticker starting 2022-06-09

fb_raw = get_massive_daily_bars("FB")
meta_raw = get_massive_daily_bars("META")

fb_raw["timestamp"] = pd.to_datetime(fb_raw["timestamp"])
meta_raw["timestamp"] = pd.to_datetime(meta_raw["timestamp"])

fb_valid = fb_raw[fb_raw["timestamp"] <= "2022-06-08"].copy()
meta_valid = meta_raw[meta_raw["timestamp"] >= "2022-06-09"].copy()

meta_clean = pd.concat([fb_valid, meta_valid], ignore_index=True)

meta_clean["ticker"] = "META"
meta_clean = meta_clean.sort_values("timestamp").reset_index(drop=True)

# Optional: convert timestamp back to date, matching your original format
meta_clean["timestamp"] = meta_clean["timestamp"].dt.date

# Save clean META file
Path("daily_data").mkdir(exist_ok=True)
meta_clean.to_csv("raw_daily_data/daily_meta.csv", index=False)

# Store in dictionary
data["META"] = meta_clean

print("Clean META saved to raw_daily_data/daily_meta.csv")

AAPL saved to raw_daily_data/daily_aapl.csv
MSFT saved to raw_daily_data/daily_msft.csv
NVDA saved to raw_daily_data/daily_nvda.csv
GOOGL saved to raw_daily_data/daily_googl.csv
AMZN saved to raw_daily_data/daily_amzn.csv
TSLA saved to raw_daily_data/daily_tsla.csv
NFLX saved to raw_daily_data/daily_nflx.csv
JPM saved to raw_daily_data/daily_jpm.csv
BAC saved to raw_daily_data/daily_bac.csv
GS saved to raw_daily_data/daily_gs.csv
MS saved to raw_daily_data/daily_ms.csv
WMT saved to raw_daily_data/daily_wmt.csv
COST saved to raw_daily_data/daily_cost.csv
HD saved to raw_daily_data/daily_hd.csv
NKE saved to raw_daily_data/daily_nke.csv
SBUX saved to raw_daily_data/daily_sbux.csv
JNJ saved to raw_daily_data/daily_jnj.csv
PFE saved to raw_daily_data/daily_pfe.csv
MRK saved to raw_daily_data/daily_mrk.csv
UNH saved to raw_daily_data/daily_unh.csv
XOM saved to raw_daily_data/daily_xom.csv
CVX saved to raw_daily_data/daily_cvx.csv
BA saved to raw_daily_data/daily_ba.csv
CAT saved to raw_daily

In [2]:
ticker_names = {
    # Big Tech / Growth
    "AAPL": "Apple Inc.",
    "MSFT": "Microsoft Corporation",
    "NVDA": "NVIDIA Corporation",
    "GOOGL": "Alphabet Inc. (Class A)",
    "AMZN": "Amazon.com, Inc.",
    "META": "Meta Platforms, Inc.",
    "TSLA": "Tesla, Inc.",
    "NFLX": "Netflix, Inc.",
    
    # Finance
    "JPM": "JPMorgan Chase & Co.",
    "BAC": "Bank of America Corporation",
    "GS": "Goldman Sachs Group, Inc.",
    "MS": "Morgan Stanley",
    
    # Consumer / Retail
    "WMT": "Walmart Inc.",
    "COST": "Costco Wholesale Corporation",
    "HD": "The Home Depot, Inc.",
    "NKE": "NIKE, Inc.",
    "SBUX": "Starbucks Corporation",
    
    # Healthcare / Pharma
    "JNJ": "Johnson & Johnson",
    "PFE": "Pfizer Inc.",
    "MRK": "Merck & Co., Inc.",
    "UNH": "UnitedHealth Group Incorporated",
    
    # Energy
    "XOM": "Exxon Mobil Corporation",
    "CVX": "Chevron Corporation",
    
    # Industrials / Transportation
    "BA": "The Boeing Company",
    "CAT": "Caterpillar Inc.",
    "GE": "General Electric Company",
    "UPS": "United Parcel Service, Inc.",
    
    # ETFs
    "SPY": "SPDR S&P 500 ETF Trust",
    "QQQ": "Invesco QQQ Trust",
    "DIA": "SPDR Dow Jones Industrial Average ETF Trust"
}

In [ ]:
df['SMA20'] = df['close'].rolling(window=20).mean()

In [2]:
from pathlib import Path
import pandas as pd

def update_daily_master(
    ticker,
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc",
    save_folder="datasets"
):
    save_path = Path(save_folder)
    save_path.mkdir(parents=True, exist_ok=True)

    master_file = save_path / f"{ticker.lower()}_daily_master.csv"

    new_df = get_massive_daily_bars(
        ticker=ticker,
        api_key_path=api_key_path,
        years=years,
        safety_days=safety_days,
        adjusted=adjusted,
        sort=sort
    )

    if master_file.exists():
        old_df = pd.read_csv(master_file)
        old_df["timestamp"] = pd.to_datetime(old_df["timestamp"]).dt.date

        combined = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined = new_df.copy()

    combined["timestamp"] = pd.to_datetime(combined["timestamp"]).dt.date

    combined = (
        combined
        .drop_duplicates(subset=["timestamp"], keep="last")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    combined.to_csv(master_file, index=False)

    print(f"{ticker} master updated: {master_file}")
    print(combined.head())
    print(combined.tail())
    print(combined.shape)
    print("Date range:", combined["timestamp"].min(), "to", combined["timestamp"].max())

    return combined

In [3]:
for ticker in tickers:
    data[ticker] = update_daily_master(ticker)

AAPL master updated: datasets/aapl_daily_master.csv
    timestamp ticker  open  high  low  close      volume  vwap  transactions
0  2021-04-22   AAPL   133   134  131    132  84,566,456   133        615670
1  2021-04-23   AAPL   132   135  132    134  78,666,779   134        519533
2  2021-04-26   AAPL   135   135  134    135  66,888,509   135        484069
3  2021-04-27   AAPL   135   135  134    134  66,015,804   135        480003
4  2021-04-28   AAPL   134   135  133    134 107,746,597   135        783355
       timestamp ticker  open  high  low  close     volume  vwap  transactions
1250  2026-04-15   AAPL   258   267  258    266 49,913,511   264        728560
1251  2026-04-16   AAPL   267   267  261    263 43,323,112   263        635080
1252  2026-04-17   AAPL   267   272  267    270 61,436,228   270        723488
1253  2026-04-20   AAPL   270   274  270    273 36,582,599   273        541032
1254  2026-04-21   AAPL   272   273  265    266 50,192,036   268        710075
(1255, 9)
Da

In [ ]:
aapl_daily = update_daily_master("AAPL")
msft_daily = update_daily_master("MSFT")
nvda_daily = update_daily_master("NVDA")

# Combine all data into one dataframe (do I need to do that here or will it be a separate notebook?)

In [3]:
import pandas as pd
import glob
import os

# 1. Define the directory path
path = 'raw_daily_data' 

# 2. Find all CSV files in that folder
all_files = glob.glob(os.path.join(path, "*.csv"))

# 3. Read each file and store in a list
df_list = []
for filename in all_files:
    df = pd.read_csv(filename)
    # Optional: Add a column to track which file the data came from
    df['source_file'] = os.path.basename(filename)
    df_list.append(df)

# 4. Combine everything into one master DataFrame
combined_df = pd.concat(df_list, ignore_index=True)

print(combined_df.head())

    timestamp ticker    open      high      low   close      volume      vwap  \
0  2021-04-28   SBUX  113.44  114.1499  111.610  112.40  14793493.0  112.4843   
1  2021-04-29   SBUX  113.21  115.1499  112.760  114.63   8936555.0  114.5165   
2  2021-04-30   SBUX  114.00  114.9500  113.490  114.49   6478714.0  114.1842   
3  2021-05-03   SBUX  115.18  116.7400  115.130  115.72   5049806.0  115.8671   
4  2021-05-04   SBUX  115.15  115.4000  113.535  114.11   6265098.0  114.2311   

   transactions     source_file  
0        177698  daily_sbux.csv  
1        115507  daily_sbux.csv  
2         75032  daily_sbux.csv  
3         67791  daily_sbux.csv  
4         85358  daily_sbux.csv  


In [4]:
combined_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,source_file
0,2021-04-28,SBUX,113.440,114.14990,111.610,112.40,1.479349e+07,112.4843,177698,daily_sbux.csv
1,2021-04-29,SBUX,113.210,115.14990,112.760,114.63,8.936555e+06,114.5165,115507,daily_sbux.csv
2,2021-04-30,SBUX,114.000,114.95000,113.490,114.49,6.478714e+06,114.1842,75032,daily_sbux.csv
3,2021-05-03,SBUX,115.180,116.74000,115.130,115.72,5.049806e+06,115.8671,67791,daily_sbux.csv
4,2021-05-04,SBUX,115.150,115.40000,113.535,114.11,6.265098e+06,114.2311,85358,daily_sbux.csv
...,...,...,...,...,...,...,...,...,...,...
37645,2026-04-21,PFE,27.520,27.62000,27.200,27.31,2.982883e+07,27.4023,136478,daily_pfe.csv
37646,2026-04-22,PFE,27.350,27.39000,26.740,26.80,3.524679e+07,26.8773,155477,daily_pfe.csv
37647,2026-04-23,PFE,26.760,26.83000,26.380,26.67,3.871841e+07,26.5511,163518,daily_pfe.csv
37648,2026-04-24,PFE,26.690,27.29903,26.580,27.00,4.250021e+07,26.9385,168756,daily_pfe.csv


In [21]:
combined_df['ticker'].value_counts()

ticker
AAPL     1255
JNJ      1255
QQQ      1255
COST     1255
MSFT     1255
AMZN     1255
TSLA     1255
CVX      1255
DIA      1255
HD       1255
UNH      1255
PFE      1255
GE       1255
CAT      1255
GOOGL    1255
JPM      1255
SBUX     1255
NFLX     1255
SPY      1255
META     1255
UPS      1255
XOM      1255
BAC      1255
NVDA     1255
GS       1255
NKE      1255
MS       1255
WMT      1255
BA       1255
MRK      1255
Name: count, dtype: int64

This is where I previously flagged that FB/META required special treatment. META lacked proper data prior to its rebrand from Facebook. Issue has been fixed.

# Volume adjusted?

In [14]:
combined_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,source_file
0,2021-04-28,SBUX,113.440,114.14990,111.610,112.40,1.479349e+07,112.4843,177698,daily_sbux.csv
1,2021-04-29,SBUX,113.210,115.14990,112.760,114.63,8.936555e+06,114.5165,115507,daily_sbux.csv
2,2021-04-30,SBUX,114.000,114.95000,113.490,114.49,6.478714e+06,114.1842,75032,daily_sbux.csv
3,2021-05-03,SBUX,115.180,116.74000,115.130,115.72,5.049806e+06,115.8671,67791,daily_sbux.csv
4,2021-05-04,SBUX,115.150,115.40000,113.535,114.11,6.265098e+06,114.2311,85358,daily_sbux.csv
...,...,...,...,...,...,...,...,...,...,...
37645,2026-04-21,PFE,27.520,27.62000,27.200,27.31,2.982883e+07,27.4023,136478,daily_pfe.csv
37646,2026-04-22,PFE,27.350,27.39000,26.740,26.80,3.524679e+07,26.8773,155477,daily_pfe.csv
37647,2026-04-23,PFE,26.760,26.83000,26.380,26.67,3.871841e+07,26.5511,163518,daily_pfe.csv
37648,2026-04-24,PFE,26.690,27.29903,26.580,27.00,4.250021e+07,26.9385,168756,daily_pfe.csv


# Manually calculated metrics

In [28]:
df_calculated_metrics = combined_df.copy()

df_calculated_metrics['close_vs_open'] = df_calculated_metrics['close'] - df_calculated_metrics['open']

df_calculated_metrics['daily_high_vs_low'] = (df_calculated_metrics['high'] - df_calculated_metrics['low'].abs())

df_calculated_metrics['close_vs_prev_day'] = df_calculated_metrics.groupby('ticker')['close'].diff()

df_calculated_metrics['open_vs_prev_close'] = (
    df_calculated_metrics['open'] 
    - df_calculated_metrics.groupby('ticker')['close'].shift(1)
)

df_calculated_metrics

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,source_file,close_vs_open,daily_high_vs_low,close_vs_prev_day,open_vs_prev_close
0,2021-04-28,SBUX,113.440,114.14990,111.610,112.40,1.479349e+07,112.4843,177698,daily_sbux.csv,-1.040,2.53990,NaN,NaN
1,2021-04-29,SBUX,113.210,115.14990,112.760,114.63,8.936555e+06,114.5165,115507,daily_sbux.csv,1.420,2.38990,2.23,0.810
2,2021-04-30,SBUX,114.000,114.95000,113.490,114.49,6.478714e+06,114.1842,75032,daily_sbux.csv,0.490,1.46000,-0.14,-0.630
3,2021-05-03,SBUX,115.180,116.74000,115.130,115.72,5.049806e+06,115.8671,67791,daily_sbux.csv,0.540,1.61000,1.23,0.690
4,2021-05-04,SBUX,115.150,115.40000,113.535,114.11,6.265098e+06,114.2311,85358,daily_sbux.csv,-1.040,1.86500,-1.61,-0.570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37645,2026-04-21,PFE,27.520,27.62000,27.200,27.31,2.982883e+07,27.4023,136478,daily_pfe.csv,-0.210,0.42000,-0.21,0.000
37646,2026-04-22,PFE,27.350,27.39000,26.740,26.80,3.524679e+07,26.8773,155477,daily_pfe.csv,-0.550,0.65000,-0.51,0.040
37647,2026-04-23,PFE,26.760,26.83000,26.380,26.67,3.871841e+07,26.5511,163518,daily_pfe.csv,-0.090,0.45000,-0.13,-0.040
37648,2026-04-24,PFE,26.690,27.29903,26.580,27.00,4.250021e+07,26.9385,168756,daily_pfe.csv,0.310,0.71903,0.33,0.020


# Same as above but with pcts

In [43]:
def add_vs_group_average(
    df,
    value_col,
    group_col='timestamp',
    new_col_name=None
):
    """
    Adds a column showing deviation from group average.

    Parameters:
    - df: DataFrame
    - value_col: column to analyze (str)
    - group_col: column to group by (default: 'timestamp')
    - new_col_name: optional custom output column name

    Returns:
    - DataFrame with new column added
    """

    if new_col_name is None:
        new_col_name = f"{value_col}_VS_{group_col.upper()}_AVG"

    group_avg = df.groupby(group_col)[value_col].transform('mean')

    df[new_col_name] = df[value_col] - group_avg

    return df



df_calculated_metrics_pct = combined_df.copy()




# Close vs Open (% change from open → close)
df_calculated_metrics_pct['close_vs_open_pct'] = (
    (df_calculated_metrics_pct['close'] - df_calculated_metrics_pct['open']) 
    / df_calculated_metrics_pct['open']
) * 100


# VS DAILY AVG
df_calculated_metrics_pct = add_vs_group_average(
    df_calculated_metrics_pct,
    value_col='close_vs_open_pct'
)





# Daily High vs Low (% range relative to low)
df_calculated_metrics_pct['daily_high_vs_low_pct'] = (
    (df_calculated_metrics_pct['high'] - df_calculated_metrics_pct['low']) 
    / df_calculated_metrics_pct['low']
) * 100

# VS DAILY AVG
df_calculated_metrics_pct = add_vs_group_average(
    df_calculated_metrics_pct,
    value_col='daily_high_vs_low_pct'
)




# Close vs Previous Day (% change from prior close)
df_calculated_metrics_pct['close_vs_prev_day_pct'] = (
    df_calculated_metrics_pct.groupby('ticker')['close']
    .pct_change()
) * 100

# VS DAILY AVG
df_calculated_metrics_pct = add_vs_group_average(
    df_calculated_metrics_pct,
    value_col='close_vs_prev_day_pct'
)




# Open vs Previous Close (% gap up/down)
df_calculated_metrics_pct['open_vs_prev_close_pct'] = (
    (df_calculated_metrics_pct['open'] - 
     df_calculated_metrics_pct.groupby('ticker')['close'].shift(1))
    / df_calculated_metrics_pct.groupby('ticker')['close'].shift(1)
) * 100

# VS DAILY AVG
df_calculated_metrics_pct = add_vs_group_average(
    df_calculated_metrics_pct,
    value_col='open_vs_prev_close_pct'
)




df_calculated_metrics_pct

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,source_file,close_vs_open_pct,close_vs_open_pct_VS_TIMESTAMP_AVG,daily_high_vs_low_pct,daily_high_vs_low_pct_VS_TIMESTAMP_AVG,close_vs_prev_day_pct,close_vs_prev_day_pct_VS_TIMESTAMP_AVG,open_vs_prev_close_pct,open_vs_prev_close_pct_VS_TIMESTAMP_AVG
0,2021-04-28,SBUX,113.440,114.14990,111.610,112.40,1.479349e+07,112.4843,177698,daily_sbux.csv,-0.916784,-0.831663,2.275692,0.762111,NaN,NaN,NaN,NaN
1,2021-04-29,SBUX,113.210,115.14990,112.760,114.63,8.936555e+06,114.5165,115507,daily_sbux.csv,1.254306,1.263327,2.119457,-0.149662,1.983986,1.136640,0.720641,-0.136456
2,2021-04-30,SBUX,114.000,114.95000,113.490,114.49,6.478714e+06,114.1842,75032,daily_sbux.csv,0.429825,0.373002,1.286457,-0.450464,-0.122132,0.442339,-0.549594,0.069258
3,2021-05-03,SBUX,115.180,116.74000,115.130,115.72,5.049806e+06,115.8671,67791,daily_sbux.csv,0.468831,0.345915,1.398419,-0.527532,1.074330,0.420341,0.602673,0.074314
4,2021-05-04,SBUX,115.150,115.40000,113.535,114.11,6.265098e+06,114.2311,85358,daily_sbux.csv,-0.903170,-0.796573,1.642665,-0.790055,-1.391289,-0.883040,-0.492568,-0.089038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37645,2026-04-21,PFE,27.520,27.62000,27.200,27.31,2.982883e+07,27.4023,136478,daily_pfe.csv,-0.763081,0.259302,1.544118,-0.959601,-0.763081,-0.064171,0.000000,-0.328283
37646,2026-04-22,PFE,27.350,27.39000,26.740,26.80,3.524679e+07,26.8773,155477,daily_pfe.csv,-2.010969,-1.957297,2.430815,0.469993,-1.867448,-2.508766,0.146466,-0.545182
37647,2026-04-23,PFE,26.760,26.83000,26.380,26.67,3.871841e+07,26.5511,163518,daily_pfe.csv,-0.336323,-0.326937,1.705838,-0.550905,-0.485075,-0.424326,-0.149254,-0.094533
37648,2026-04-24,PFE,26.690,27.29903,26.580,27.00,4.250021e+07,26.9385,168756,daily_pfe.csv,1.161484,1.039164,2.705154,0.726957,1.237345,1.132569,0.074991,0.094711


# Close_Vs_Open_Pct Agg

# SMAs

Dilemma - With a strict 20 day SMA, the first 20 days of the dataset will be dropped for lack of available data. And so on with larger spans.

I will make two versions - one with the strict limit that will result in some NaNs, and another that uses whatever data available.

In [25]:
df_sma_strict = combined_df.copy()
df_sma_loose = combined_df.copy()

df_sma_strict['close_SMA10'] = df_sma_strict['close'].rolling(window=10).mean()
df_sma_strict['close_SMA20'] = df_sma_strict['close'].rolling(window=20).mean()
df_sma_strict['close_SMA50'] = df_sma_strict['close'].rolling(window=50).mean()
df_sma_strict['close_SMA100'] = df_sma_strict['close'].rolling(window=100).mean()

df_sma_loose['close_SMA10'] = df_sma_loose['close'].rolling(window=10, min_periods=1).mean()
df_sma_loose['close_SMA20'] = df_sma_loose['close'].rolling(window=20, min_periods=1).mean()
df_sma_loose['close_SMA50'] = df_sma_loose['close'].rolling(window=50, min_periods=1).mean()
df_sma_loose['close_SMA100'] = df_sma_loose['close'].rolling(window=100, min_periods=1).mean()

In [26]:
df_sma_strict

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,source_file,close_SMA10,close_SMA20,close_SMA50,close_SMA100
0,2021-04-28,SBUX,113.440,114.14990,111.610,112.40,1.479349e+07,112.4843,177698,daily_sbux.csv,NaN,NaN,NaN,NaN
1,2021-04-29,SBUX,113.210,115.14990,112.760,114.63,8.936555e+06,114.5165,115507,daily_sbux.csv,NaN,NaN,NaN,NaN
2,2021-04-30,SBUX,114.000,114.95000,113.490,114.49,6.478714e+06,114.1842,75032,daily_sbux.csv,NaN,NaN,NaN,NaN
3,2021-05-03,SBUX,115.180,116.74000,115.130,115.72,5.049806e+06,115.8671,67791,daily_sbux.csv,NaN,NaN,NaN,NaN
4,2021-05-04,SBUX,115.150,115.40000,113.535,114.11,6.265098e+06,114.2311,85358,daily_sbux.csv,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37645,2026-04-21,PFE,27.520,27.62000,27.200,27.31,2.982883e+07,27.4023,136478,daily_pfe.csv,27.286,27.4680,27.2490,26.4443
37646,2026-04-22,PFE,27.350,27.39000,26.740,26.80,3.524679e+07,26.8773,155477,daily_pfe.csv,27.219,27.4600,27.2440,26.4551
37647,2026-04-23,PFE,26.760,26.83000,26.380,26.67,3.871841e+07,26.5511,163518,daily_pfe.csv,27.164,27.4295,27.2252,26.4647
37648,2026-04-24,PFE,26.690,27.29903,26.580,27.00,4.250021e+07,26.9385,168756,daily_pfe.csv,27.172,27.4010,27.2106,26.4773


In [27]:
df_sma_loose

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,source_file,close_SMA10,close_SMA20,close_SMA50,close_SMA100
0,2021-04-28,SBUX,113.440,114.14990,111.610,112.40,1.479349e+07,112.4843,177698,daily_sbux.csv,112.400,112.4000,112.4000,112.4000
1,2021-04-29,SBUX,113.210,115.14990,112.760,114.63,8.936555e+06,114.5165,115507,daily_sbux.csv,113.515,113.5150,113.5150,113.5150
2,2021-04-30,SBUX,114.000,114.95000,113.490,114.49,6.478714e+06,114.1842,75032,daily_sbux.csv,113.840,113.8400,113.8400,113.8400
3,2021-05-03,SBUX,115.180,116.74000,115.130,115.72,5.049806e+06,115.8671,67791,daily_sbux.csv,114.310,114.3100,114.3100,114.3100
4,2021-05-04,SBUX,115.150,115.40000,113.535,114.11,6.265098e+06,114.2311,85358,daily_sbux.csv,114.270,114.2700,114.2700,114.2700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37645,2026-04-21,PFE,27.520,27.62000,27.200,27.31,2.982883e+07,27.4023,136478,daily_pfe.csv,27.286,27.4680,27.2490,26.4443
37646,2026-04-22,PFE,27.350,27.39000,26.740,26.80,3.524679e+07,26.8773,155477,daily_pfe.csv,27.219,27.4600,27.2440,26.4551
37647,2026-04-23,PFE,26.760,26.83000,26.380,26.67,3.871841e+07,26.5511,163518,daily_pfe.csv,27.164,27.4295,27.2252,26.4647
37648,2026-04-24,PFE,26.690,27.29903,26.580,27.00,4.250021e+07,26.9385,168756,daily_pfe.csv,27.172,27.4010,27.2106,26.4773
